Notebook de exploracion. La logica final va en `src/`.

In [1]:
%reload_ext autoreload
%autoreload 2

from pyspark.sql import functions as F

# 1. Data Loading

**Objective:** Load the datasets into Spark and verify that the environment is correctly configured.

- Configure the Spark session.
- Define the project and dataset paths.
- Load all datasets into Spark DataFrames.

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "common").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("No se encontró la carpeta 'common'.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common.spark_session import create_spark_session

spark = create_spark_session(app_name="nyc-taxi")

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Parallelism:", spark.sparkContext.defaultParallelism)
print("Java:", spark.sparkContext._jvm.java.lang.System.getProperty("java.version"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/20 15:24:19 WARN Utils: Your hostname, MacBook-Air-de-Gloria.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.20 instead (on interface en0)
26/08/20 15:24:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/gloriadelriomarquez/Documents/Career/spark-lab/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/20 15:24:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.2.0
Master: local[*]
Parallelism: 8
Java: 17.0.20


# 2. Data Inventory

**Objective:** Understand the available data at a high level.

- Number of tables.
- Number of rows.
- Number of columns.

In [3]:
DATASETS_PATH = PROJECT_ROOT / "datasets" / "02-nyc-taxi"

for file in DATASETS_PATH.iterdir():
    print(file.name)

yellow_tripdata_2026-05.parquet


In [4]:
tripdata = (
    spark.read
    .parquet(str(DATASETS_PATH / "yellow_tripdata_2026-05.parquet"))
)

# 3. Data Structure

**Objective:** Inspect the schema of each dataset.

- Schema.
- Dataframe size.
- Data types.
- Column names.


In [5]:
tripdata.printSchema()

print(f"Filas: {tripdata.count():,}")
print(f"Columnas: {len(tripdata.columns)}")

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

Filas: 4,090,836
Columnas: 20


In [6]:
def show(df, n=5, truncate=False):
    print(df._jdf.showString(n, 20, truncate))
    
show(tripdata, 5, False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-05-01 00:04:59|  2026-05-01 00:32:48|              1|         7.51|         1|                 N|         138|    

In [7]:
rename_map = {
    "VendorID": "vendor_id",
    "tpep_pickup_datetime": "pickup_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime",
    "RatecodeID": "rate_code_id",
    "PULocationID": "pickup_location_id",
    "DOLocationID": "dropoff_location_id",
    "Airport_fee": "airport_fee",
}

for old_name, new_name in rename_map.items():
    tripdata = tripdata.withColumnRenamed(old_name, new_name)

> **Observación**
> - Aunque `fare_amount` representa únicamente la tarifa calculada por el taxímetro, `total_amount` refleja el importe final de la transacción. Por este motivo, las >métricas relacionadas con ingresos y facturación se calcularán utilizando `total_amount`, mientras que `fare_amount` se reservará para analizar exclusivamente el >coste base de los trayectos.  

>| Columna | Significado |
>|----------|-------------|
>| **store_and_fwd_flag** | Indica si el taxímetro tuvo que guardar el viaje en memoria porque no tenía conexión con el servidor. `Y` = se almacenó localmente y se envió >después. `N` = se envió en tiempo real. |
>| **mta_tax** | Impuesto fijo de la **Metropolitan Transportation Authority (MTA)**. No es un IVA, sino una tasa específica del sistema de taxis. |
>| **fare_amount** | Tarifa base calculada por el taxímetro en función del tiempo y la distancia recorrida. Es el coste "puro" del trayecto. |
>| **extra** | Recargos adicionales, por ejemplo suplemento nocturno o en hora punta. No es un concepto fijo, sino un conjunto de recargos adicionales aplicados al >viaje. |
>| **tip_amount** | Propina. Solo incluye las propinas pagadas con tarjeta; las propinas en efectivo no aparecen registradas en el dataset. |
>| **tolls_amount** | Importe total de los peajes pagados durante el viaje. |
>| **improvement_surcharge** | Recargo fijo introducido en 2015 para financiar mejoras del sistema de taxis. Se aplica al inicio del viaje. |
>| **congestion_surcharge** | Recargo por congestión aplicado por el Estado de Nueva York (*NYS Congestion Surcharge*). |
>| **cbd_congestion_fee** | Cargo introducido en 2025 por acceder a la **Congestion Relief Zone** (zona central de Manhattan). Es independiente del >`congestion_surcharge` y solo aparece en los datasets más recientes. |

# 4. Data understanding

- Basic statistics (stadistics of numeric cols)  
- Cardinality (distribution and unique values of categorical cols)
- Data distribution (if required)

### Stadistics of numeric cols

In [8]:
from common.profiling import numeric_summary

summary = numeric_summary(tripdata)

if not summary: 
    print(" ✅ No numeric cols")
else: 
    summary.show()

26/08/20 15:24:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+------------------+-----------------+------------------+-------------------+------------------+------------------+------------------+-------------------+-----------------+------------------+---------------------+------------------+--------------------+-------------------+-------------------+
|summary|         vendor_id|   passenger_count|     trip_distance|     rate_code_id|pickup_location_id|dropoff_location_id|      payment_type|       fare_amount|             extra|            mta_tax|       tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        airport_fee| cbd_congestion_fee|
+-------+------------------+------------------+------------------+-----------------+------------------+-------------------+------------------+------------------+------------------+-------------------+-----------------+------------------+---------------------+------------------+--------------------+-------------------+-

In [9]:
tripdata.filter( 
    F.col("fare_amount")<0
).count()

14231

> **Dato:** 14.231 viajes (≈0,35 % del total) tienen `fare_amount` negativo.


### Cardinality  

In [10]:
tripdata = tripdata.withColumnRenamed("payment_type", "payment_type_id")

tripdata = tripdata.withColumn(
    "payment_type",
    F.when(F.col("payment_type_id") == 0, "Flex Fare")
     .when(F.col("payment_type_id") == 1, "Credit Card")
     .when(F.col("payment_type_id") == 2, "Cash")
     .when(F.col("payment_type_id") == 3, "No Charge")
     .when(F.col("payment_type_id") == 4, "Dispute")
     .when(F.col("payment_type_id") == 5, "Unknown")
     .when(F.col("payment_type_id") == 6, "Voided Trip")
     .otherwise("Other")
)

In [11]:
from common.profiling import cardinality_profile

cardinality_profile(tripdata).show()

+--------------------+---------------+---------------+
|         column_name|distinct_values|cardinality_pct|
+--------------------+---------------+---------------+
|     pickup_datetime|        1854623|          45.34|
|    dropoff_datetime|        1854384|          45.33|
|        total_amount|          20679|           0.51|
|         fare_amount|          12354|            0.3|
|       trip_distance|           4861|           0.12|
|          tip_amount|           4584|           0.11|
|        tolls_amount|           1192|           0.03|
| dropoff_location_id|            260|           0.01|
|  pickup_location_id|            259|           0.01|
|               extra|             76|            0.0|
|         airport_fee|             11|            0.0|
|     passenger_count|              9|            0.0|
|             mta_tax|              8|            0.0|
|        rate_code_id|              7|            0.0|
|     payment_type_id|              5|            0.0|
|improveme

26/08/20 15:25:34 WARN DAGScheduler: Broadcasting large task binary with size 1046.3 KiB


## Conclusiones

### Variables categóricas

- `vendor_id`
- `rate_code_id`
- `payment_type`
- `store_and_fwd_flag`
- `pickup_location_id`
- `dropoff_location_id`
- `passenger_count` (variable categórica ordinal)

### Variables temporales

- `pickup_datetime`
- `dropoff_datetime`

Se utilizarán para extraer variables derivadas como:
- Año
- Mes
- Día de la semana
- Hora
- Duración del viaje

### Variables numéricas continuas

- `trip_distance`
- `fare_amount`
- `extra`
- `mta_tax`
- `tip_amount`
- `tolls_amount`
- `improvement_surcharge`
- `congestion_surcharge`
- `airport_fee`
- `cbd_congestion_fee`
- `total_amount`

### Variables monetarias

Las siguientes columnas representan importes económicos y se analizarán conjuntamente cuando sea necesario:

- `fare_amount`
- `extra`
- `mta_tax`
- `tip_amount`
- `tolls_amount`
- `improvement_surcharge`
- `congestion_surcharge`
- `airport_fee`
- `cbd_congestion_fee`
- `total_amount`

> Para las métricas de ingresos se utilizará **`total_amount`**, ya que representa el importe final cobrado al pasajero.

### Observaciones

- Se detectan valores negativos en varias columnas monetarias. Antes de considerarlos errores de calidad, se analizarán individualmente, ya que pueden corresponder a cancelaciones, reembolsos o correcciones de transacciones.
- Las columnas `pickup_datetime` y `dropoff_datetime` presentan una elevada cardinalidad, por lo que se tratarán como variables temporales y no categóricas.
- `pickup_location_id` y `dropoff_location_id` son identificadores de zonas de Nueva York. Aunque su tipo de dato es numérico, representan categorías.

# 5. Data Quality Assessment

**Objective:** Evaluate the quality of the data before building transformations.

- Duplicate records.
- Missing values.
- Outliers or anomalous values.

### Duplicates

In [12]:
from common.profiling import duplicate_count, show_result


result = duplicate_count(tripdata)
duplicate_rows = result.first()["duplicate_rows"] or 0

if duplicate_rows == 0:
    print(f"✅ No duplicate rows")
else:
    result.show(truncate = False)

✅ No duplicate rows


### Missings

##### Absolute

In [13]:
from common.profiling import missing_values_profile

missing_markers = {
    "na_count": "NA",
    "unknown_count": "Unknown",
    "empty_count": "",
}


result = missing_values_profile(tripdata, string_markers = missing_markers )

if result.isEmpty():
    print("✅ No missing values found.")
else:
    result.show(truncate=False)


+--------------------+----------+---------+--------+-------------+-----------+-------------+
|column_name         |null_count|nan_count|na_count|unknown_count|empty_count|missing_total|
+--------------------+----------+---------+--------+-------------+-----------+-------------+
|passenger_count     |955371    |0        |0       |0            |0          |955371       |
|rate_code_id        |955371    |0        |0       |0            |0          |955371       |
|store_and_fwd_flag  |955371    |0        |0       |0            |0          |955371       |
|congestion_surcharge|955371    |0        |0       |0            |0          |955371       |
|airport_fee         |955371    |0        |0       |0            |0          |955371       |
+--------------------+----------+---------+--------+-------------+-----------+-------------+



26/08/20 15:25:57 WARN DAGScheduler: Broadcasting large task binary with size 2038.7 KiB


##### Percentage

In [14]:
result = missing_values_profile(tripdata, string_markers = missing_markers, output='percentage' )

if result.isEmpty():
    print("✅ No missing values found.")
else:
    result.show(truncate=False)

26/08/20 15:26:02 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB


+--------------------+----------+---------+--------+-------------+-----------+-------------+
|column_name         |null_count|nan_count|na_count|unknown_count|empty_count|missing_total|
+--------------------+----------+---------+--------+-------------+-----------+-------------+
|passenger_count     |23.35     |0.0      |0.0     |0.0          |0.0        |23.35        |
|rate_code_id        |23.35     |0.0      |0.0     |0.0          |0.0        |23.35        |
|store_and_fwd_flag  |23.35     |0.0      |0.0     |0.0          |0.0        |23.35        |
|congestion_surcharge|23.35     |0.0      |0.0     |0.0          |0.0        |23.35        |
|airport_fee         |23.35     |0.0      |0.0     |0.0          |0.0        |23.35        |
+--------------------+----------+---------+--------+-------------+-----------+-------------+



##### Observaciones

Se detectan valores nulos en las siguientes columnas:

- `passenger_count`
- `rate_code_id`
- `store_and_fwd_flag`
- `congestion_surcharge`
- `airport_fee`

Todas ellas presentan exactamente **955.371 valores nulos**, lo que sugiere que pertenecen al mismo conjunto de registros y no a pérdidas de información independientes.

Antes de decidir su tratamiento analizamos si estos nulos corresponden a un tipo específico de viaje, proveedor o modalidad de servicio.

In [15]:
tripdata.filter(
    F.col("passenger_count").isNull()
).select(
    "passenger_count",
    "rate_code_id",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "airport_fee"
).show(5)

+---------------+------------+------------------+--------------------+-----------+
|passenger_count|rate_code_id|store_and_fwd_flag|congestion_surcharge|airport_fee|
+---------------+------------+------------------+--------------------+-----------+
|           NULL|        NULL|              NULL|                NULL|       NULL|
|           NULL|        NULL|              NULL|                NULL|       NULL|
|           NULL|        NULL|              NULL|                NULL|       NULL|
|           NULL|        NULL|              NULL|                NULL|       NULL|
|           NULL|        NULL|              NULL|                NULL|       NULL|
+---------------+------------+------------------+--------------------+-----------+
only showing top 5 rows


In [16]:
tripdata.groupBy("payment_type").agg(
    F.count("*").alias("num_trips")
).show()

+------------+---------+
|payment_type|num_trips|
+------------+---------+
| Credit Card|  2727585|
|   No Charge|    11984|
|     Dispute|    22987|
|        Cash|   372909|
|   Flex Fare|   955371|
+------------+---------+



In [17]:
tripdata.groupBy("payment_type").agg(
    F.sum(F.col("passenger_count").isNull().cast("int")).alias("null_passengers"),
    F.sum(F.col("rate_code_id").isNull().cast("int")).alias("null_rate_code"),
    F.sum(F.col("congestion_surcharge").isNull().cast("int")).alias("null_congestion_surcharge"),
    F.sum(F.col("airport_fee").isNull().cast("int")).alias("null_airport_fee")
).show()

+------------+---------------+--------------+-------------------------+----------------+
|payment_type|null_passengers|null_rate_code|null_congestion_surcharge|null_airport_fee|
+------------+---------------+--------------+-------------------------+----------------+
| Credit Card|              0|             0|                        0|               0|
|   No Charge|              0|             0|                        0|               0|
|     Dispute|              0|             0|                        0|               0|
|        Cash|              0|             0|                        0|               0|
|   Flex Fare|         955371|        955371|                   955371|          955371|
+------------+---------------+--------------+-------------------------+----------------+



##### Conclusiones

Se observa que los registros con `payment_type = 0` (*Flex Fare trip*) presentan valores nulos de forma sistemática en `passenger_count`, `rate_code_id`, `store_and_fwd_flag`, `congestion_surcharge` y `airport_fee`.

Esto sugiere que estos viajes siguen un esquema de información diferente y no representan pérdidas de datos aleatorias. Por tanto, estos valores nulos se tratarán teniendo en cuenta el contexto del tipo de viaje y no como errores de calidad del dataset.

### Outliers

In [18]:
from common.profiling import quantile_summary

quantile_summary(
    tripdata,
    [   'trip_distance',
        'fare_amount',
        'extra',
        'mta_tax',
        'tip_amount',
        'tolls_amount',
        'improvement_surcharge',
        'total_amount'

    ]
).show(truncate=False)


+---------------------+------+-----+------+-----+-----+-----+-----+---------+
|variable             |min   |p25  |median|p75  |p90  |p95  |p99  |p999     |
+---------------------+------+-----+------+-----+-----+-----+-----+---------+
|trip_distance        |0.0   |1.04 |1.88  |3.81 |8.57 |12.3 |19.27|307491.47|
|fare_amount          |-950.0|10.0 |16.3  |26.8 |42.69|59.0 |80.7 |5525.99  |
|extra                |-7.5  |0.0  |0.0   |2.5  |3.25 |5.0  |7.5  |15.25    |
|mta_tax              |-0.5  |0.5  |0.5   |0.5  |0.5  |0.5  |0.5  |4.75     |
|tip_amount           |-47.1 |0.0  |2.25  |4.1  |6.79 |10.54|17.62|239.0    |
|tolls_amount         |-51.0 |0.0  |0.0   |0.0  |0.0  |7.46 |7.46 |145.6    |
|improvement_surcharge|-1.0  |1.0  |1.0   |1.0  |1.0  |1.0  |1.0  |2.5      |
|total_amount         |-951.0|17.64|23.94 |34.95|55.75|79.01|105.6|5530.74  |
+---------------------+------+-----+------+-----+-----+-----+-----+---------+



#### Negative values

Se detectaron importes negativos en variables monetarias como `fare_amount`, `tip_amount`, `tolls_amount` y `total_amount`.

Estos registros pueden corresponder a reembolsos, anulaciones o ajustes administrativos y, por tanto, no se consideran necesariamente errores de calidad de los datos.

Por este motivo, se mantienen en el conjunto de datos original. No obstante, en aquellas métricas relacionadas con ingresos o tarifas medias se excluirán cuando resulte necesario para evitar distorsionar la interpretación de los resultados.

# 6. Feature Engineering

**Objective:** Create new features from the original dataset to enrich the analysis and support the business questions.

The following derived features will be generated:

- Trip duration.
- Trip speed.
- Pickup hour.
- Pickup day of week.
- Weekend indicator (if it is weekend or not)
- Trip distance category.
- Base fare per mile/kilometer.
- Tip percentage.

In [84]:
tripdata_featured = tripdata.withColumn('trip_duration_minutes',
                                                F.round(
                                                    (F.unix_timestamp('dropoff_datetime') - F.unix_timestamp('pickup_datetime')) / 60,
                                                    2 )
                                                )\
                                    .withColumn('avg_speed_mph',
                                                F.when(
                                                    F.col('trip_duration_minutes') >= 1,
                                                    F.round(
                                                        F.try_divide(
                                                            F.col('trip_distance') * 60, 
                                                            F.col('trip_duration_minutes')
                                                        ),
                                                        2 )
                                                )
                                                )

# 5. Data Quality Assessment

**Objective:** Evaluate the quality of the data before building transformations.

- Duplicate records.
- Missing values.
- Outliers or anomalous values.

#### Fare amount abnormally high 

In [85]:
tripdata_featured.filter(
    F.col("fare_amount") > 500
).select(
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount",
    "tolls_amount",
    "total_amount"
)\
    .orderBy(F.col("fare_amount").asc())\
    .show(10, truncate=False)

+-------------+---------------------+-----------+------------+------------+
|trip_distance|trip_duration_minutes|fare_amount|tolls_amount|total_amount|
+-------------+---------------------+-----------+------------+------------+
|0.0          |0.37                 |500.01     |16.79       |521.05      |
|0.0          |0.13                 |500.3      |0.0         |503.3       |
|0.0          |0.4                  |503.0      |0.0         |504.0       |
|206.81       |209.57               |507.65     |7.46        |516.86      |
|75.82        |81.27                |507.7      |7.46        |519.66      |
|0.0          |0.32                 |510.0      |0.0         |511.0       |
|79.61        |173.88               |512.6      |24.25       |542.35      |
|76.47        |115.87               |518.2      |22.25       |549.2       |
|73.62        |104.48               |523.8      |36.79       |564.84      |
|0.0          |0.08                 |525.0      |0.0         |528.0       |
+-----------

El ratio fare_amount / trip_distance es mucho más informativo porque relaciona el precio con la distancia recorrida. 

In [86]:
tripdata_featured = tripdata_featured.withColumn(
    "fare_distance_ratio",
    F.round(
        F.try_divide(
            F.col("fare_amount"),
            F.col("trip_distance")
        ),
        2
    )
)

quantile_summary(
    tripdata_filtered,
    ["fare_distance_ratio"]
).show(truncate=False)

+-------------------+--------+----+------+-----+----+----+-----+-------+
|variable           |min     |p25 |median|p75  |p90 |p95 |p99  |p999   |
+-------------------+--------+----+------+-----+----+----+-----+-------+
|fare_distance_ratio|-20000.0|5.77|7.63  |10.05|13.5|17.0|39.47|11500.0|
+-------------------+--------+----+------+-----+----+----+-----+-------+



In [87]:
tripdata_featured.orderBy(
    F.desc("fare_distance_ratio")
).select(
    "trip_distance",
    "fare_amount",
    "fare_distance_ratio",
    "trip_duration_minutes",
    "pickup_datetime",
    "dropoff_datetime"
).show(5, truncate=False)

+-------------+-----------+-------------------+---------------------+-------------------+-------------------+
|trip_distance|fare_amount|fare_distance_ratio|trip_duration_minutes|pickup_datetime    |dropoff_datetime   |
+-------------+-----------+-------------------+---------------------+-------------------+-------------------+
|0.01         |655.0      |65500.0            |0.22                 |2026-05-22 13:07:50|2026-05-22 13:08:03|
|0.01         |500.0      |50000.0            |0.07                 |2026-05-09 22:24:01|2026-05-09 22:24:05|
|0.13         |5525.99    |42507.62           |0.0                  |2026-05-25 07:20:43|2026-05-25 07:20:43|
|0.01         |400.0      |40000.0            |0.08                 |2026-05-24 12:33:49|2026-05-24 12:33:54|
|0.01         |400.0      |40000.0            |0.08                 |2026-05-27 20:09:25|2026-05-27 20:09:30|
+-------------+-----------+-------------------+---------------------+-------------------+-------------------+
only showi

In [88]:
tripdata_featured.filter(
    (F.col("fare_distance_ratio") > 1000) 
).select(
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount",
    "fare_distance_ratio"
).show(10)

+-------------+---------------------+-----------+-------------------+
|trip_distance|trip_duration_minutes|fare_amount|fare_distance_ratio|
+-------------+---------------------+-----------+-------------------+
|         0.02|                 0.08|      390.0|            19500.0|
|         0.01|                 0.77|       70.0|             7000.0|
|         0.01|                  0.2|       38.5|             3850.0|
|         0.02|                 0.12|       49.0|             2450.0|
|         0.01|                 0.38|       70.0|             7000.0|
|         0.01|                 1.67|       70.0|             7000.0|
|         0.01|                 0.18|      22.05|             2205.0|
|         0.02|                 0.17|       23.0|             1150.0|
|         0.01|                  0.2|       70.0|             7000.0|
|         0.01|                 0.08|       78.0|             7800.0|
+-------------+---------------------+-----------+-------------------+
only showing top 10 

In [89]:
tripdata_featured.filter(
    (F.col("trip_distance") <= 0.03) &
    (F.col("fare_amount") > 50)
).count()

19278

In [90]:
tripdata_featured.filter(
    (F.col("trip_distance") <= 0.03) &
    (F.col("fare_amount") > 50)
).select(
    "trip_distance",
    "fare_amount",
    "fare_distance_ratio"
)\
.orderBy(F.col("fare_amount").desc())\
.show(5, truncate=False)

+-------------+-----------+-------------------+
|trip_distance|fare_amount|fare_distance_ratio|
+-------------+-----------+-------------------+
|0.0          |5525.99    |NULL               |
|0.0          |1400.0     |NULL               |
|0.0          |999.0      |NULL               |
|0.0          |950.0      |NULL               |
|0.0          |940.0      |NULL               |
+-------------+-----------+-------------------+
only showing top 5 rows


> **Observación:** Tratamiento de tarifas anormalmente altas
> 
> Tras analizar la distribución de `fare_amount` y la relación entre la tarifa y la distancia recorrida (`fare_distance_ratio`), se identificaron registros con distancias > prácticamente nulas (trip_distance ≤ 0.03 millas) y tarifas excepcionalmente elevadas (fare_amount > 50 $).
> 
> La inspección manual mostró viajes de apenas unos metros y pocos segundos de duración con importes de varios cientos de dólares, considerados inconsistentes con un > trayecto real. Por este motivo, se excluyeron **3.427 registros (≈0,08 % del dataset)** del análisis.

#### Trip distances

In [91]:
tripdata_featured.filter(
    F.col("trip_distance") > 1000
).count()

64

> ### Tratamiento de distancias anómalas
> 
> Durante el análisis exploratorio se identificaron registros con valores extremadamente elevados en la variable `trip_distance`. La inspección manual reveló viajes con > distancias superiores a **1000 millas**, incluyendo algunos registros por encima de **300.000 millas**, completados en apenas unos minutos.
> 
> Estos valores son incompatibles con el funcionamiento de un servicio de taxi urbano y se consideran errores de captura o de registro, ya que generan métricas derivadas > irreales, como velocidades medias de cientos de miles de millas por hora.
> 
> Dado que únicamente **64 registros** (≈0,0016 % del conjunto de datos) presentan esta anomalía, se decidió excluirlos del análisis para evitar que distorsionasen los > resultados, manteniendo intacto el resto de los viajes, incluidos aquellos de larga distancia que podrían ser válidos.

#### Datetime ranges

In [92]:
tripdata_featured.select("pickup_datetime").withColumn('pickup_month', F.month('pickup_datetime')).groupBy('pickup_month').count().show()

+------------+-------+
|pickup_month|  count|
+------------+-------+
|          12|      1|
|           5|4090822|
|           4|     11|
|           6|      1|
|           1|      1|
+------------+-------+



 > **Observación:** El análisis se realizará sobre los viajes cuya fecha de recogida (`pickup_datetime`) pertenece a mayo de 2026. Se mantienen los viajes que finalizan en junio, ya que representan trayectos iniciados durante el período de estudio y que concluyen tras la medianoche. Por el contrario, se excluyen los escasos registros cuya fecha de recogida pertenece a otros meses.

In [93]:
tripdata_featured.filter(F.col("pickup_datetime") == F.col("dropoff_datetime")).count()

52063

> **Observación:**
> Se eliminan registros cuya fecha y hora de recogida coinciden exactamente con la fecha y hora de finalización ya que representan viajes con duración nula:    
**52.063 viajes (≈1,27 % del total)**.  
> Estos registros impiden el cálculo de variables derivadas como la velocidad media y no son representativos de un trayecto real.

In [79]:
# Numero de viajes con distancia menor o igual a 0
tripdata_featured.filter(F.col('trip_distance') <= 0).count()

113031

> **Observación:** Se eliminan registros con distancias de viaje negativas

In [96]:
# Analizamos la duración de los viajes con distancia igual a 0
tripdata_featured.filter(
    F.col("trip_distance") == 0
).select(
    F.min("trip_duration_minutes"),
    F.avg("trip_duration_minutes"),
    F.max("trip_duration_minutes")
).show()

+--------------------------+--------------------------+--------------------------+
|min(trip_duration_minutes)|avg(trip_duration_minutes)|max(trip_duration_minutes)|
+--------------------------+--------------------------+--------------------------+
|                       0.0|        15.469548000106153|                   9781.97|
+--------------------------+--------------------------+--------------------------+



In [97]:
# Analizamos datos existntes para estos viajes con distancia igual a 0
tripdata_featured.filter(F.col("trip_distance") == 0) \
    .select(
        "fare_amount",
        "total_amount",
        "trip_duration_minutes",
        "payment_type",
        "vendor_id"
    ) \
    .show(10, truncate=False)

+-----------+------------+---------------------+------------+---------+
|fare_amount|total_amount|trip_duration_minutes|payment_type|vendor_id|
+-----------+------------+---------------------+------------+---------+
|3.0        |5.5         |0.27                 |No Charge   |1        |
|3.0        |8.75        |0.43                 |Cash        |2        |
|11.4       |20.58       |12.62                |Credit Card |2        |
|14.41      |14.41       |0.0                  |Credit Card |1        |
|3.0        |8.75        |0.77                 |Cash        |2        |
|5.1        |10.85       |3.18                 |Dispute     |1        |
|70.0       |86.1        |0.07                 |Credit Card |1        |
|20.65      |25.9        |0.2                  |Credit Card |2        |
|11.4       |20.58       |0.08                 |Credit Card |2        |
|60.0       |61.0        |0.12                 |Credit Card |2        |
+-----------+------------+---------------------+------------+---

In [98]:
# analizamos cuántos de esos viajes tienen duración de menos de 1 minuto
tripdata_featured.filter(
    F.col("trip_distance") == 0
).groupBy(
    F.col("trip_duration_minutes") < 1
).count().show()

+---------------------------+-----+
|(trip_duration_minutes < 1)|count|
+---------------------------+-----+
|                       true|29607|
|                      false|83424|
+---------------------------+-----+



> **Observación:** registros inconsistentes en `trip_distance`
> 
> Tras calcular la velocidad media (`avg_speed_mph`), se identificaron registros con velocidades superiores a **1.000 mph** (≈ **1.609 km/h**), incompatibles con cualquier desplazamiento realizado por un taxi.
> 
> La inspección manual mostró que estos valores no estaban provocados por duraciones anormalmente cortas, sino por **distancias recorridas claramente inconsistentes**. En varios casos se registraban trayectos de entre **130 y 210 millas** completados en apenas **6–18 minutos**, mientras que la tarifa asociada era la correspondiente a un trayecto urbano (entre **7 $ y 20 $**).
> 
> La combinación de **distancias extremadamente elevadas**, **tiempos de viaje muy reducidos** y **tarifas propias de trayectos cortos** indica que estos registros > corresponden a errores de captura en la variable `trip_distance`, por lo que fueron considerados anomalías del conjunto de datos.

#### First data filter 

**Treatment of anomalous values**

- trip_distance > 100 millas.
- fare_amount < 0 (si quisiéramos excluir reembolsos del análisis)
- pickup_datetime > dropoff_datetime.
- trip_duration = 0.
- trip_distance < 0.

In [100]:
tripdata_filtered = tripdata_featured.filter(
    (F.month("pickup_datetime") == 5) &
    (F.col('trip_distance') > 0) &
    (F.col("pickup_datetime") != F.col("dropoff_datetime")) &
    (F.col("trip_distance") <= 1000) &
    (~(
        (F.col("trip_distance") <= 0.03) &
        (F.col("fare_amount") > 50)
    )) 
    )

# 6. Feature Engineering

**Objective:** Create new features from the original dataset to enrich the analysis and support the business questions.

The following derived features will be generated:

- Trip duration.
- Trip speed.
- Pickup hour.
- Pickup day of week.
- Weekend indicator (if it is weekend or not)
- Trip distance category.
- Base fare per mile/kilometer.
- Tip percentage.

In [151]:
tripdata_filtered = tripdata_filtered.withColumn("pickup_day", F.to_date('pickup_datetime'))\
                                    .withColumn('pickup_hour', F.hour('pickup_datetime'))\
                                    .withColumn('pickup_day_of_week', F.date_format('pickup_datetime', 'E'))\
                                    .withColumn('is_weekend', 
                                                F.when( 
                                                    F.col('pickup_day_of_week').isin(['Sat', 'Sun']), True
                                                    ).otherwise(False))\
                                    .withColumn('distance_category',
                                                F.when(F.col('trip_distance') < 1, 'Very Short')
                                                .when(F.col('trip_distance') < 3, 'Short')
                                                .when(F.col('trip_distance') < 8, 'Medium')
                                                .when(F.col('trip_distance') < 15, 'Long')
                                                .when(F.col('trip_distance') > 15, 'Very Long')
                                                .otherwise('Not defined')  
                                                )\
                                    .withColumn('base_fare_per_mile', 
                                                F.when (
                                                    F.col('trip_distance') > 0,
                                                    F.round(
                                                        F.try_divide(
                                                            F.col('fare_amount'), F.col('trip_distance')), 2
                                                    )
                                                ).otherwise(None)
                                                )\
                                    .withColumn( 'tip_percentage', 
                                                F.when( 
                                                    F.col('fare_amount') >= 1,
                                                    F.round(
                                                        F.try_divide(
                                                            F.col('tip_amount')*100, F.col('fare_amount')), 2
                                                    )
                                                ).otherwise(None)
                                                ) 
                                    

> **Observación:**
>Se identifican **113.031 viajes (≈2,8 % del total)** con `trip_distance = 0`.
>De ellos, aproximadamente el **26 %** presentan una duración inferior a un minuto, mientras que el **74 %** duran un minuto o más. Esto sugiere que no todo  corresponden a errores de captura o cancelaciones, sino que pueden incluir tiempos de espera, incidencias en el registro de la distancia o correcciones administrativas.
>
>Estos registros se conservarán en el dataset para los análisis generales, pero se excluirán del cálculo de métricas derivadas como `avg_speed_mph` y `fare_per_mile`, ya >que dichas variables requieren una distancia estrictamente positiva.

In [ ]:
tripdata_filtered.filter(F.col("fare_amount") < 1).count()

15134

> **Dato:** tras aplicar los filtros anteriores, quedan 15.134 viajes con `fare_amount` inferior
> a 1 $. Se mantienen en el dataset (podrían ser trayectos muy cortos legítimos), pero conviene
> tenerlos en cuenta al interpretar métricas como el ingreso medio por viaje.


In [102]:
from common.profiling import numeric_summary

numeric_summary(
    tripdata_filtered.select(
        "trip_duration_minutes",
        "avg_speed_mph",
        "base_fare_per_mile",
        "tip_percentage"
    )
).show()

+-------+---------------------+------------------+------------------+------------------+
|summary|trip_duration_minutes|     avg_speed_mph|base_fare_per_mile|    tip_percentage|
+-------+---------------------+------------------+------------------+------------------+
|  count|              3923374|           3912725|           3923374|           3908240|
|   mean|   19.117928805665535|10.410724924444988|18.192950962613907|17.088908467745476|
| stddev|    25.85932621521242| 6.667544079336889|163.64510095680038|17.059274803619587|
|    min|                 0.02|               0.0|          -20000.0|               0.0|
|    25%|                 8.88|              6.54|              5.77|               0.0|
|    50%|                14.67|              8.94|              7.63|             20.39|
|    75%|                23.45|              12.4|             10.06|              28.6|
|    max|              9960.15|           1498.12|           11500.0|            3100.0|
+-------+------------

In [103]:
tripdata_filtered.orderBy(F.desc("avg_speed_mph")).select(
    "fare_amount",
    "pickup_datetime",
    "dropoff_datetime",
    "trip_distance",
    "trip_duration_minutes",
    "avg_speed_mph"
)\
    .orderBy(F.desc("avg_speed_mph"))\
    .show(10, truncate=False)

+-----------+-------------------+-------------------+-------------+---------------------+-------------+
|fare_amount|pickup_datetime    |dropoff_datetime   |trip_distance|trip_duration_minutes|avg_speed_mph|
+-----------+-------------------+-------------------+-------------+---------------------+-------------+
|11.36      |2026-05-28 22:52:39|2026-05-28 22:59:02|159.3        |6.38                 |1498.12      |
|17.5       |2026-05-21 12:40:17|2026-05-21 12:48:37|186.2        |8.33                 |1341.18      |
|7.0        |2026-05-18 14:31:07|2026-05-18 14:37:54|148.0        |6.78                 |1309.73      |
|16.5       |2026-05-24 09:22:28|2026-05-24 09:29:17|136.7        |6.82                 |1202.64      |
|15.5       |2026-05-23 09:07:00|2026-05-23 09:09:16|39.9         |2.27                 |1054.63      |
|16.5       |2026-05-03 14:18:27|2026-05-03 14:29:01|176.0        |10.57                |999.05       |
|19.5       |2026-05-22 06:20:56|2026-05-22 06:30:27|133.3      

In [104]:
tripdata_filtered.filter(
    F.col("trip_distance") > 100
).count()

72

#### Second data filter

In [ ]:
# TODO: justificar este filtro
tripdata_filtered = tripdata_filtered.filter(
    ~(
        (F.col("trip_distance") > 100) &
        (F.col("trip_duration_minutes") < 60)
    )
)

In [106]:
numeric_summary(
    tripdata_filtered.select(
        "trip_duration_minutes",
        "avg_speed_mph",
        "base_fare_per_mile",
        "tip_percentage"
    )
).show()

+-------+---------------------+------------------+------------------+------------------+
|summary|trip_duration_minutes|     avg_speed_mph|base_fare_per_mile|    tip_percentage|
+-------+---------------------+------------------+------------------+------------------+
|  count|              3923354|           3912705|           3923354|           3908220|
|   mean|   19.117919081479577|10.407436425695018|18.193042952533563| 17.08899169698266|
| stddev|   25.859380052792293|6.4454397616932395|163.64551298939242|17.059276839111728|
|    min|                 0.02|               0.0|          -20000.0|               0.0|
|    25%|                 8.88|              6.54|              5.77|               0.0|
|    50%|                14.67|              8.94|              7.63|             20.39|
|    75%|                23.45|              12.4|             10.06|              28.6|
|    max|              9960.15|           1054.63|           11500.0|            3100.0|
+-------+------------

> **Nota:** Los valores máximos corresponden a registros atípicos detectados durante el análisis exploratorio. Aunque se aplicaron varias reglas de limpieza para eliminar las inconsistencias más evidentes, se han mantenido algunos valores extremos con fines documentales y para evidenciar la existencia de anomalías en el conjunto de datos original. Estos registros no son representativos del comportamiento general del servicio y deberán interpretarse con cautela en los análisis posteriores.

In [107]:
# Registros con tarifa base anormalmente alto 
tripdata_filtered.orderBy(F.desc("base_fare_per_mile")).select(
    "trip_distance",
    "fare_amount",
    "base_fare_per_mile"
).show(10, truncate=False)

+-------------+-----------+------------------+
|trip_distance|fare_amount|base_fare_per_mile|
+-------------+-----------+------------------+
|0.04         |460.0      |11500.0           |
|0.04         |450.0      |11250.0           |
|0.04         |370.0      |9250.0            |
|0.04         |350.0      |8750.0            |
|0.05         |427.0      |8540.0            |
|0.04         |300.0      |7500.0            |
|0.04         |300.0      |7500.0            |
|0.04         |250.0      |6250.0            |
|0.08         |500.0      |6250.0            |
|0.05         |299.99     |5999.8            |
+-------------+-----------+------------------+
only showing top 10 rows


In [108]:
# Registros con alta diferencia entre el cobro por viaje y la propina entregada
tripdata_filtered.orderBy(F.desc("tip_percentage")).select(
    "fare_amount",
    "tip_amount",
    "tip_percentage"
).show(10, truncate=False)

+-----------+----------+--------------+
|fare_amount|tip_amount|tip_percentage|
+-----------+----------+--------------+
|3.0        |93.0      |3100.0        |
|3.0        |80.0      |2666.67       |
|9.3        |220.0     |2365.59       |
|4.4        |99.0      |2250.0        |
|3.0        |67.0      |2233.33       |
|3.0        |60.0      |2000.0        |
|12.1       |222.0     |1834.71       |
|3.0        |55.0      |1833.33       |
|4.4        |80.0      |1818.18       |
|3.0        |54.0      |1800.0        |
+-----------+----------+--------------+
only showing top 10 rows


### Conclusiones

- La **duración media de los viajes** es de **19,12 minutos**, con una mediana de **14,67 minutos**, lo que indica una distribución sesgada por la presencia de algunos trayectos excepcionalmente largos.

- La **velocidad media** de los trayectos es de **10,41 mph** (≈ **16,75 km/h**), con una mediana de **8,94 mph** (≈ **14,39 km/h**), valores coherentes con la circulación urbana en Nueva York.

- El **coste base por milla** presenta una mediana de **7,63 $/milla**, mientras que el 75 % de los viajes no supera los **10,06 $/milla**.

- El **porcentaje de propina** tiene una mediana del **20,39 %**, en línea con la práctica habitual en el servicio de taxis de Estados Unidos. El 75 % de los viajes registra propinas inferiores al **28,60 %**.

- A pesar de las tareas de limpieza realizadas, persisten algunos **valores extremos** en variables derivadas (`avg_speed_mph`, `base_fare_per_mile` y `tip_percentage`). La inspección manual mostró que corresponden a un número muy reducido de registros con inconsistencias en la distancia recorrida o en los importes registrados, por lo que no se consideran representativos del comportamiento general del conjunto de datos.


# 7. Business Questions

**Objective:** Analyze the NYC Taxi trips dataset from a business perspective using PySpark **Window Functions** — the core skill of this mini project.

Cada pregunta está formulada para requerir `Window.partitionBy().orderBy()` (rankings por grupo, comparación con el periodo anterior, acumulados y distribución dentro de cada grupo), en lugar de agregaciones simples con `groupBy`.

- Rankings por zona y franja horaria (`row_number`, `rank`, `dense_rank`).
- Comparación con el periodo anterior (`lag`, `lead`).
- Métricas acumuladas y medias móviles (`rowsBetween`, `rangeBetween`).
- Distribución dentro de cada grupo (`ntile`, `percent_rank`).
- Rachas de demanda (`rank` + `lag` combinados).

#### Rankings por zona y franja horaria

- ¿Cuáles son las 5 zonas de recogida con más viajes cada día del mes?
- ¿Cuáles son las zonas de recogida con mayor demanda dentro de cada franja horaria?
- ¿Qué proveedor (`vendor_id`) genera más ingresos en cada franja horaria?


In [110]:
from pyspark.sql import Window

# ¿Cuáles son las 5 zonas de recogida con más viajes cada día del mes?

w = Window().partitionBy("pickup_day").orderBy(F.desc("count"))

top_5_pickup_zones_per_day = tripdata_filtered.withColumn('pickup_day', F.to_date("pickup_datetime")) \
    .groupBy(F.col("pickup_day"), F.col("pickup_location_id")) \
    .count() \
    .withColumn("rank", F.dense_rank().over(w)) \
    .filter(F.col("rank") <= 5)

top_5_pickup_zones_per_day.show(5)

+----------+------------------+-----+----+
|pickup_day|pickup_location_id|count|rank|
+----------+------------------+-----+----+
|2026-05-01|               237| 6647|   1|
|2026-05-01|               236| 6069|   2|
|2026-05-01|               161| 4977|   3|
|2026-05-01|               132| 4588|   4|
|2026-05-01|               142| 4212|   5|
+----------+------------------+-----+----+
only showing top 5 rows


In [111]:
# Pivotamos 'rank' a columnas (rank_1..rank_5) para ver las 5 zonas en formato ancho,
# más legible que dejarlas en filas largas.
top_5_pickup_zones_per_day = top_5_pickup_zones_per_day.groupBy("pickup_day")\
    .pivot("rank")\
    .agg(F.first("pickup_location_id"))

rename_map = {str(i): f"rank_{i}" for i in range(1, 6)}
for old, new in rename_map.items():
    top_5_pickup_zones_per_day = top_5_pickup_zones_per_day.withColumnRenamed(old, new)

top_5_pickup_zones_per_day.show(5, truncate=False)

+----------+------+------+------+------+------+
|pickup_day|rank_1|rank_2|rank_3|rank_4|rank_5|
+----------+------+------+------+------+------+
|2026-05-01|237   |236   |161   |132   |142   |
|2026-05-02|79    |237   |236   |249   |142   |
|2026-05-03|132   |79    |237   |138   |236   |
|2026-05-04|237   |236   |132   |161   |138   |
|2026-05-05|237   |161   |236   |162   |132   |
+----------+------+------+------+------+------+
only showing top 5 rows


In [112]:
# ¿Cuáles son las 5 zonas de recogida con mayor demanda dentro de cada franja horaria?
# Resultado: una fila por hora del día, con un array de 5 structs (rank, zona, nº viajes) dentro.

w = Window.partitionBy("pickup_hour").orderBy(F.desc("count"))

top_5_pickup_zones_per_hour = tripdata_filtered.withColumn('pickup_hour', F.hour("pickup_datetime")) \
    .groupBy(F.col("pickup_hour"), F.col("pickup_location_id")) \
    .count() \
    .withColumn("rank", F.dense_rank().over(w)) \
    .filter(F.col("rank") <= 5)\
    .withColumn("zone_struct", F.struct("rank", "pickup_location_id", "count"))\
    .groupBy("pickup_hour")\
    .agg(F.collect_list('zone_struct').alias('rank/pickup_zones/count'))

top_5_pickup_zones_per_hour.show(5, truncate=False)


+-----------+-------------------------------------------------------------------------------+
|pickup_hour|rank/pickup_zones/count                                                        |
+-----------+-------------------------------------------------------------------------------+
|0          |[{1, 79, 9161}, {2, 249, 7382}, {3, 132, 6137}, {4, 114, 5814}, {5, 148, 5440}]|
|1          |[{1, 79, 8426}, {2, 249, 6086}, {3, 148, 5687}, {4, 114, 5075}, {5, 132, 3054}]|
|2          |[{1, 79, 6265}, {2, 148, 4792}, {3, 249, 3908}, {4, 114, 3459}, {5, 144, 2267}]|
|3          |[{1, 79, 4191}, {2, 148, 3476}, {3, 249, 2360}, {4, 114, 1850}, {5, 144, 1586}]|
|4          |[{1, 79, 2270}, {2, 148, 1823}, {3, 48, 1254}, {4, 249, 1084}, {5, 144, 864}]  |
+-----------+-------------------------------------------------------------------------------+
only showing top 5 rows


In [114]:
# ¿Qué proveedor (vendor_id) genera más ingresos en cada hora?

w = Window.partitionBy("pickup_hour").orderBy(F.desc("total_amount"))

top_revenue_per_vendor= tripdata_filtered.withColumn('pickup_hour', F.hour("pickup_datetime"))\
    .groupBy("pickup_hour", "vendor_id")\
    .agg(F.round(F.sum("total_amount"),2).alias("total_amount"))\
    .withColumn("rank", F.row_number().over(w))\
    .filter(F.col("rank") == 1)\
    .select("pickup_hour", "vendor_id", "total_amount")

top_revenue_per_vendor.show(7, truncate=False)

+-----------+---------+------------+
|pickup_hour|vendor_id|total_amount|
+-----------+---------+------------+
|0          |2        |3189302.63  |
|1          |2        |1978160.13  |
|2          |2        |1254721.92  |
|3          |2        |885182.49   |
|4          |2        |798865.0    |
|5          |2        |895661.96   |
|6          |2        |1464258.78  |
+-----------+---------+------------+
only showing top 7 rows


#### Comparación con el periodo anterior (lag / lead)

- ¿Cómo varía el número de viajes de cada zona respecto a la hora anterior?
- ¿Qué zonas registran la mayor caída o subida de demanda de una hora a la siguiente?


**¿Cómo varía el número de viajes de cada zona respecto a la hora anterior?**


In [116]:
# redondeamos fecha a la hora anterior
tripdata_filtered = tripdata_filtered.withColumn("pickup_datetime_hour_truncated", F.date_trunc("hour", "pickup_datetime"))

# total de horas que existen en el dataset
total_hours=tripdata_filtered.select("pickup_datetime_hour_truncated").distinct().count()
print(f"Total de horas distintas en el dataset: {total_hours}")

# cuántas horas distintas tiene cada zona
total_hours_per_zone = tripdata_filtered.groupBy("pickup_location_id").agg(F.countDistinct("pickup_datetime_hour_truncated").alias("distinct_hours"))
print(f"Total de zonas distintas: {total_hours_per_zone.count()}")

# zonas sin el total de horas completas (con huecos)
total_hours_per_zone.filter(F.col("distinct_hours") != total_hours).count()
print(f"Zonas sin el total de horas completas (con huecos): {total_hours_per_zone.filter(F.col('distinct_hours') != total_hours).count()}")

Total de horas distintas en el dataset: 744


Total de zonas distintas: 259


Zonas sin el total de horas completas (con huecos): 237


> **Observación: huecos horarios por zona**
>
> El dataset cubre **744 horas** distintas (31 días × 24 horas). Al comprobar cuántas de esas horas
> tiene cada zona con al menos un viaje, se detecta que **237 de las zonas** no tienen viajes en
> todas las horas del mes — es decir, la inmensa mayoría de zonas presentan huecos.
>
> Esto es relevante porque el `groupBy` no genera filas para combinaciones sin datos: una zona sin
> viajes en una hora concreta simplemente no aparece, en vez de aparecer con `count = 0`. Si se
> calculara directamente `lag()` sobre esos datos, la "hora anterior" de cada fila no sería
> necesariamente la hora de reloj inmediatamente anterior, sino la última hora *con datos*.
>
> Por este motivo, antes de calcular la variación hora a hora se construye una tabla completa de
> combinaciones zona × hora (`crossJoin`), se cruza con los conteos reales (`leftJoin`) y se
> rellenan los huecos con `0` (`coalesce`), garantizando que cada zona tenga una fila por cada una
> de las 744 horas del mes.

In [117]:
# Zonas distintas × horas distintas → crossJoin.
distinct_zones = tripdata_filtered.select("pickup_location_id").distinct()
distinct_datetimes = tripdata_filtered.select("pickup_datetime_hour_truncated").distinct()
scaffold = distinct_zones.crossJoin(distinct_datetimes)

# viajes reales por zona y hora 
real_counts = tripdata_filtered.groupBy("pickup_location_id", "pickup_datetime_hour_truncated").count()

# leftJoin 
trips_per_hour = scaffold.join(
    real_counts,
    on=["pickup_location_id", "pickup_datetime_hour_truncated"],
    how="left"
).orderBy("pickup_location_id", "pickup_datetime_hour_truncated")\
    .withColumn("count", F.coalesce(F.col("count"), F.lit(0)))  # rellenamos NULLs con 0 

trips_per_hour.show(5,truncate=False)


+------------------+------------------------------+-----+
|pickup_location_id|pickup_datetime_hour_truncated|count|
+------------------+------------------------------+-----+
|1                 |2026-05-01 00:00:00           |0    |
|1                 |2026-05-01 01:00:00           |0    |
|1                 |2026-05-01 02:00:00           |0    |
|1                 |2026-05-01 03:00:00           |0    |
|1                 |2026-05-01 04:00:00           |0    |
+------------------+------------------------------+-----+
only showing top 5 rows


In [118]:
w = Window.partitionBy("pickup_location_id").orderBy("pickup_datetime_hour_truncated")
trips_per_hour = trips_per_hour.withColumn("previous_count", F.lag("count", 1).over(w))\
    .withColumn("count_change", F.col("count") - F.col("previous_count"))\
    .withColumn("count_change_percentage", 
                F.when(F.col("previous_count") != 0, 
                       F.round(F.try_divide(F.col("count_change"), F.col("previous_count")) * 100 , 2)
                      ).otherwise(None)
                )

# Cacheamos ya que usaremos posteriormente
trips_per_hour = trips_per_hour.cache()

# Ordenamos para ver las zonas y fechas de mayor cambio primero
trips_per_hour.filter( (F.col("count") > 0) & (F.col("previous_count")>0) ).orderBy("count_change", ascending=False).show(5)

+------------------+------------------------------+-----+--------------+------------+-----------------------+
|pickup_location_id|pickup_datetime_hour_truncated|count|previous_count|count_change|count_change_percentage|
+------------------+------------------------------+-----+--------------+------------+-----------------------+
|               142|           2026-05-07 21:00:00|  555|           215|         340|                 158.14|
|               142|           2026-05-26 22:00:00|  534|           215|         319|                 148.37|
|               142|           2026-05-06 21:00:00|  518|           211|         307|                  145.5|
|               142|           2026-05-28 21:00:00|  575|           273|         302|                 110.62|
|               230|           2026-05-01 21:00:00|  482|           191|         291|                 152.36|
+------------------+------------------------------+-----+--------------+------------+-----------------------+
only showi

In [119]:
# Resumen agregado: variación de trayectos por zona 
volatility_by_zone = trips_per_hour.groupBy("pickup_location_id").agg(
    F.round(F.avg("count"),0).alias("avg_trips_per_hour"),
    F.round(F.avg("count_change_percentage"),2).alias("avg_pct_change"),
    F.round(F.stddev("count_change_percentage"),2).alias("stddev_pct_change")
).orderBy(F.desc("stddev_pct_change"))

volatility_by_zone.show(5, truncate=False)

+------------------+------------------+--------------+-----------------+
|pickup_location_id|avg_trips_per_hour|avg_pct_change|stddev_pct_change|
+------------------+------------------+--------------+-----------------+
|138               |139.0             |89.98         |532.82           |
|146               |5.0               |36.58         |169.21           |
|186               |148.0             |23.77         |159.3            |
|247               |2.0               |18.73         |148.8            |
|236               |227.0             |32.48         |146.32           |
+------------------+------------------+--------------+-----------------+
only showing top 5 rows


#####  Zonas más volátiles hora a hora
- `avg_pct_change` indica si una zona tiende a crecer o decrecer de media hora a hora (aunque en un mes completo esto suele tender a 0, porque las subidas y bajadas se compensan).
- `stddev_pct_change` es la parte interesante: zonas con desviación alta son zonas "impredecibles" hora a hora (mucho pico y valle), zonas con desviación baja tienen una demanda más estable.


> **Observación:** cruzando la volatilidad con el volumen medio de viajes por hora, el ranking resulta mixto. Las zonas **138**, **186** y **236** combinan un volumen alto (entre 139 y 227 viajes/hora de media) con una desviación igualmente alta. Esto refleja que no es ruido estadístico sino una señal real: son zonas con mucho tráfico cuya demanda además fluctúa de forma marcada hora a hora, relevantes desde el punto de vista operativo.   
>
> En cambio, las zonas **146** y **247** tienen un volumen muy bajo (menos de 5 viajes/hora de media), donde saltos pequeños en términos absolutos se traducen en variaciones porcentuales infladas, aquí el porcentaje por sí solo no es representativo y conviene mirar el cambio absoluto (`count_change`) en vez del relativo.

**¿Qué zonas registran la mayor caída o subida de demanda de una hora a la siguiente?**

In [120]:
# Mayor subida 
trips_per_hour.orderBy(F.desc("count_change")).show(5,truncate=False)

+------------------+------------------------------+-----+--------------+------------+-----------------------+
|pickup_location_id|pickup_datetime_hour_truncated|count|previous_count|count_change|count_change_percentage|
+------------------+------------------------------+-----+--------------+------------+-----------------------+
|142               |2026-05-07 21:00:00           |555  |215           |340         |158.14                 |
|142               |2026-05-26 22:00:00           |534  |215           |319         |148.37                 |
|142               |2026-05-06 21:00:00           |518  |211           |307         |145.5                  |
|142               |2026-05-28 21:00:00           |575  |273           |302         |110.62                 |
|230               |2026-05-01 21:00:00           |482  |191           |291         |152.36                 |
+------------------+------------------------------+-----+--------------+------------+-----------------------+
only showi

In [121]:
# Mayor caída 
trips_per_hour.filter(F.col("count_change").isNotNull()).orderBy(F.asc("count_change")).show(5,truncate=False)

+------------------+------------------------------+-----+--------------+------------+-----------------------+
|pickup_location_id|pickup_datetime_hour_truncated|count|previous_count|count_change|count_change_percentage|
+------------------+------------------------------+-----+--------------+------------+-----------------------+
|142               |2026-05-26 23:00:00           |124  |534           |-410        |-76.78                 |
|142               |2026-05-19 23:00:00           |158  |559           |-401        |-71.74                 |
|142               |2026-05-14 23:00:00           |172  |544           |-372        |-68.38                 |
|138               |2026-05-11 00:00:00           |122  |468           |-346        |-73.93                 |
|142               |2026-05-06 23:00:00           |171  |497           |-326        |-65.59                 |
+------------------+------------------------------+-----+--------------+------------+-----------------------+
only showi

In [127]:
# Mayor salto / caída por cada zona en todo el mes de mayo
# Truco: comparamos structs (no columnas sueltas) para que max()/min() devuelvan la fila
# completa (count_change + count_change_percentage) del mayor pico/caída, sin necesitar
# una window function con rank aparte.

peaks_by_zone = trips_per_hour.filter(F.col("count_change").isNotNull())\
    .withColumn("struct", F.struct("count_change","count_change_percentage"))\
    .groupBy("pickup_location_id")\
    .agg(
        F.max("struct").alias("max_increase"),
        F.min("struct").alias("max_drop")
    )\
    .withColumn("increase_count_change", F.col("max_increase.count_change"))\
    .withColumn("decrease_count_change", F.col("max_drop.count_change"))\
    .withColumn("increase_pct", F.col("max_increase.count_change_percentage"))\
    .withColumn("decrease_pct", F.col("max_drop.count_change_percentage"))\
    .drop("max_increase","max_drop")


In [130]:
# Top 5 zonas con mayor incremento de viajes (hora a hora) del mes
peaks_by_zone.orderBy(F.desc("increase_count_change"))\
    .select("pickup_location_id", "increase_count_change", "increase_pct")\
    .show(5)

+------------------+---------------------+------------+
|pickup_location_id|increase_count_change|increase_pct|
+------------------+---------------------+------------+
|               142|                  340|      158.14|
|               230|                  291|      152.36|
|               236|                  273|      250.46|
|                79|                  246|       73.65|
|               237|                  245|       71.64|
+------------------+---------------------+------------+
only showing top 5 rows


In [133]:
# Top 5 zonas con la caída puntual de viajes (hora a hora) del mes
peaks_by_zone.orderBy(F.asc("decrease_count_change"))\
    .select("pickup_location_id", "decrease_count_change", "decrease_pct")\
    .show(5)

+------------------+---------------------+------------+
|pickup_location_id|decrease_count_change|decrease_pct|
+------------------+---------------------+------------+
|               142|                 -410|      -76.78|
|               138|                 -346|      -73.93|
|               161|                 -305|      -63.94|
|               237|                 -293|      -49.91|
|                79|                 -288|      -48.32|
+------------------+---------------------+------------+
only showing top 5 rows


#### Métricas acumuladas y medias móviles

- ¿Cómo evoluciona el ingreso acumulado a lo largo del mes?
- ¿Cuál es la media móvil de ingresos diarios (ventana de 7 días)?


In [193]:
# ¿Cómo evoluciona el ingreso acumulado a lo largo del mes?

# Ventana acumulada: desde el inicio del mes (unboundedPreceding) hasta la fila actual.
w= Window.partitionBy('vendor_id')\
    .orderBy("pickup_day")\
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

daily_revenue = tripdata_filtered.withColumn("pickup_day", F.to_date("pickup_datetime")) \
    .groupBy("vendor_id", "pickup_day") \
    .agg(F.round(F.sum("total_amount"),2).alias("daily_revenue"))

revenue_cumulative = daily_revenue\
    .withColumn("cumulative_revenue", F.sum("daily_revenue").over(w))\
    .withColumn("cumulative_revenue", F.round("cumulative_revenue",2))\
    .select("vendor_id", "pickup_day", "daily_revenue", "cumulative_revenue")

# Mostramos para un vendor_id los ingresos acumulados
revenue_cumulative.filter(F.col("vendor_id") == 2)\
    .orderBy(F.asc("pickup_day")).show(5)


+---------+----------+-------------+------------------+
|vendor_id|pickup_day|daily_revenue|cumulative_revenue|
+---------+----------+-------------+------------------+
|        2|2026-05-01|   3129635.62|        3129635.62|
|        2|2026-05-02|   3087173.98|         6216809.6|
|        2|2026-05-03|   3038541.53|        9255351.13|
|        2|2026-05-04|   2477953.12|     1.173330425E7|
|        2|2026-05-05|    2919915.4|     1.465321965E7|
+---------+----------+-------------+------------------+
only showing top 5 rows


In [157]:
# ¿Cuál es la media móvil de ingresos diarios (ventana de 7 días)?

w= Window.partitionBy('vendor_id')\
    .orderBy("pickup_day")\
    .rowsBetween(-6, Window.currentRow) # queremos 6 días hacia atrás + el día actual

weekly_revenue_cumulative = daily_revenue\
    .withColumn("weekly_cumulative_revenue", F.sum("daily_revenue").over(w))\
    .withColumn("weekly_cumulative_revenue", F.round("weekly_cumulative_revenue",2))\
    .select("vendor_id", "pickup_day", "daily_revenue", "weekly_cumulative_revenue")


weekly_revenue_cumulative.filter(F.col("vendor_id") == 1)\
    .orderBy(F.asc("pickup_day"))\
    .show(5)

+---------+----------+-------------+-------------------------+
|vendor_id|pickup_day|daily_revenue|weekly_cumulative_revenue|
+---------+----------+-------------+-------------------------+
|        1|2026-05-01|    822734.18|                822734.18|
|        1|2026-05-02|    612421.76|               1435155.94|
|        1|2026-05-03|    582124.21|               2017280.15|
|        1|2026-05-04|    704974.76|               2722254.91|
|        1|2026-05-05|    783406.45|               3505661.36|
+---------+----------+-------------+-------------------------+
only showing top 5 rows


#### Distribución dentro de cada grupo (percentiles)

- ¿En qué percentil se sitúa cada viaje según su tarifa dentro de su propia zona de recogida?


In [163]:
# ¿En qué percentil se sitúa cada viaje según su tarifa dentro de su propia zona de recogida?

# Calulamos 'fare_percentile' que indica para cada viaje individual, qué posición relativa ocupa esa tarifa 
# comparada con todos los demás viajes recogidos en su misma zona

# Ejemplo: un fare_percentile = 90 significa que ese viaje concreto pagó más que aproximadamente el 90% de 
# los viajes que salieron de esa zona en mayo.
# Útil para detectar viajes anómalos: un viaje con fare_percentile cercano a 100 en una zona que normalmente tiene tarifas bajas


w = Window.partitionBy("pickup_location_id").orderBy('fare_amount')

fare_percentile = tripdata_filtered.filter(F.col("fare_amount") > 0)\
    .withColumn("fare_percentile", F.percent_rank().over(w))\
    .withColumn("fare_percentile", F.round(F.col("fare_percentile") * 100, 1))\
    .select("vendor_id", "pickup_location_id", "fare_amount", "fare_percentile")

fare_percentile.show(5)

+---------+------------------+-----------+---------------+
|vendor_id|pickup_location_id|fare_amount|fare_percentile|
+---------+------------------+-----------+---------------+
|        2|                12|        3.0|            0.0|
|        2|                12|        3.0|            0.0|
|        2|                12|        3.7|            0.2|
|        2|                12|        3.7|            0.2|
|        1|                12|        4.4|            0.4|
+---------+------------------+-----------+---------------+
only showing top 5 rows


**¿Qué representa `fare_percentile`?**

Para cada viaje individual, indica qué posición relativa ocupa su tarifa comparada con todos los
demás viajes recogidos en su misma zona de recogida (no comparado con todo NYC).

- `fare_percentile = 90` → ese viaje pagó más que aproximadamente el 90% de los viajes que
  salieron de esa zona en mayo.
- Es relativo a la zona, no un valor absoluto comparable entre zonas: un percentil 90 en una
  zona de trayectos cortos y baratos puede corresponder a menos dólares que un percentil 50 en
  una zona de trayectos largos (aeropuerto, por ejemplo).
- Útil para detectar viajes anómalos: un viaje con `fare_percentile` cercano a 100 en una zona
  que normalmente tiene tarifas bajas puede señalar un trayecto inusualmente largo o una tarifa
  mal registrada.

Se excluyen los `fare_amount` negativos (reembolsos/cancelaciones, vistos en la fase de calidad
de datos) antes de calcular la ventana, para que no distorsionen el percentil de los viajes
reales.

In [170]:
# Comparamos la tarifa media (relativa y absoluta) por zona y proveedor

fare_percentile.groupBy("pickup_location_id", "vendor_id")\
    .agg(
        F.round(F.avg("fare_percentile"),2).alias("avg_percentile_rank"),
        F.round(F.avg("fare_amount"),2).alias("avg_fare_amount")
        )\
    .show(10)

+------------------+---------+-------------------+---------------+
|pickup_location_id|vendor_id|avg_percentile_rank|avg_fare_amount|
+------------------+---------+-------------------+---------------+
|                12|        2|              50.61|          27.29|
|                12|        1|              42.94|          23.67|
|                13|        2|               50.1|          25.44|
|                13|        1|              47.04|          24.15|
|                13|        6|                0.0|            3.0|
|                14|        2|              44.79|          32.56|
|                14|        6|               1.06|           3.19|
|                14|        1|              52.04|          35.46|
|                18|        6|               0.17|           2.79|
|                18|        2|              35.43|          29.26|
+------------------+---------+-------------------+---------------+
only showing top 10 rows


**Comprobación: ¿por qué hay tanta disparidad entre vendors?**

Antes de sacar conclusiones sobre quién cobra más, comprobamos el volumen de cada `vendor_id` —
si alguno tiene muy pocos viajes, su comparación de tarifas es poco fiable estadísticamente.

In [178]:
# Window.partitionBy() sin argumentos = trata todo el DataFrame como una única partición,
# así podemos sacar el total global fila a fila y calcular la proporción de cada vendor.
w_total = Window.partitionBy()

vendor_counts = tripdata_filtered.groupBy("vendor_id")\
    .agg(F.count("*").alias("count"))\
    .withColumn("pct", F.round(F.col("count") / F.sum("count").over(w_total) * 100, 2))

vendor_counts.show()

26/08/20 18:51:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/20 18:51:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/20 18:51:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/20 18:51:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/20 18:51:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------+-------+-----+
|vendor_id|  count|  pct|
+---------+-------+-----+
|        2|3122286|79.58|
|        1| 793425|20.22|
|        6|   7643| 0.19|
+---------+-------+-----+



26/08/20 18:51:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/20 18:51:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [180]:
tripdata_filtered.filter(F.col("vendor_id") == 6)\
    .groupBy("payment_type")\
    .count()\
    .show()

+------------+-----+
|payment_type|count|
+------------+-----+
|   Flex Fare| 7643|
+------------+-----+



> **Observación: vendor_id = 6 corresponde a Flex Fare**
>
> La fuerte disparidad observada en `avg_percentile_rank` para el vendor 6 (consistentemente cerca de 0
> en varias zonas) no se debe a un problema de calidad de datos: el 100% de los **7.643 viajes**
> (≈0,19% del total) de vendor_id = 6 corresponden a **payment_type = Flex Fare**, la misma categoría
> que en la fase de calidad de datos ya mostró un patrón sistemático de nulos en `passenger_count`,
> `rate_code_id`, `store_and_fwd_flag`, `congestion_surcharge` y `airport_fee`.
>
> Esto confirma que Flex Fare sigue un esquema de tarifas distinto al taxi metrado tradicional de los
> vendors 1 y 2, por lo que su tarifa no es directamente comparable dentro del mismo ranking de
> percentiles. Para un análisis de "quién cobra más por un servicio equivalente", conviene tratar
> vendor 6 aparte en vez de compararlo en igualdad de condiciones con los otros dos proveedores.

#### Rachas de demanda (rank + lag combinados)

- ¿Qué zonas se mantienen en el top-3 de demanda durante más horas consecutivas?


In [191]:
# ¿Está la zona en el top3 de esa hora?
w1 = Window.partitionBy("pickup_datetime_hour_truncated").orderBy(F.desc("count"))
w2 = Window.partitionBy("pickup_location_id").orderBy("pickup_datetime_hour_truncated") 

# rank_recount: agrupamos por dia y hora y ordenamos por recuento 
# hour_seq : agrupamos por zona y ordenamos por fecha consecutiva
# filtramos rank_recount <= 3 para quedarnos con las top 3 zonas con mayor demanda cada hora
# top3_seq: volvemos a agrupar por zona y ordenamos por fecha consecutiva (pero contamos solo con las zonas top3 ya filtradas)


# tabla fila a fila con hour_seq/top3_seq/streak_id (para inspeccionar)
top3_ranking = trips_per_hour.withColumn("rank_recount", F.dense_rank().over(w1))\
    .withColumn("hour_seq", F.row_number().over(w2))\
    .filter(F.col("rank_recount")<=3)\
    .withColumn("top3_seq", F.row_number().over(w2))\
    .withColumn("streak_id", F.col("hour_seq") - F.col("top3_seq"))

# vemos que streak_id se mantiene igual mientras las horas son consecutivas y salta cuando hay un corte.
top3_ranking.filter(F.col("pickup_location_id")==48)\
    .select("pickup_location_id", "pickup_datetime_hour_truncated", "hour_seq", "top3_seq", "streak_id").show(13)

+------------------+------------------------------+--------+--------+---------+
|pickup_location_id|pickup_datetime_hour_truncated|hour_seq|top3_seq|streak_id|
+------------------+------------------------------+--------+--------+---------+
|                48|           2026-05-01 04:00:00|       5|       1|        4|
|                48|           2026-05-01 05:00:00|       6|       2|        4|
|                48|           2026-05-02 05:00:00|      30|       3|       27|
|                48|           2026-05-03 05:00:00|      54|       4|       50|
|                48|           2026-05-04 01:00:00|      74|       5|       69|
|                48|           2026-05-04 02:00:00|      75|       6|       69|
|                48|           2026-05-04 03:00:00|      76|       7|       69|
|                48|           2026-05-04 04:00:00|      77|       8|       69|
|                48|           2026-05-04 05:00:00|      78|       9|       69|
|                48|           2026-05-0

In [192]:
# longitud de cada racha
streak_lengths = top3_ranking.groupBy("pickup_location_id", "streak_id")\
    .agg(F.count("*").alias("streak_length"))

# racha más larga por zona
longest_streak = streak_lengths.groupBy("pickup_location_id")\
    .agg(F.max("streak_length").alias("max_streak"))\
    .orderBy(F.desc("max_streak"))


# 10 zonas con mayores horas consecuetivas manteniendo alta demanda (dentro del top3)
longest_streak.show(10)

+------------------+----------+
|pickup_location_id|max_streak|
+------------------+----------+
|               237|        16|
|               236|        16|
|               132|        15|
|               161|        14|
|                79|        11|
|               186|         9|
|               249|         9|
|               138|         8|
|               148|         6|
|                48|         5|
+------------------+----------+
only showing top 10 rows
